# Monte Carlo verification of Black-Scholes pricing
### Baseline verification, financial consistency, and delta-hedging validation

Before a pricing model can be used to measure model risk, its implementation has to be trusted. This notebook verifies the analytical Black-Scholes call price from `bs_model.py` against an independent risk-neutral Monte Carlo estimator, and then uses the same simulation engine to test the model's stronger claim: that its delta hedge replicates the option's payoff.

The analysis proceeds in four stages:

1. **Baseline verification.** Under constant volatility, simulated terminal prices must reproduce the theoretical lognormal moments, and Monte Carlo call prices must agree with the exact Black-Scholes prices within their sampling error. A convergence study confirms that the error shrinks at the expected rate.
2. **Financial consistency.** Both pricing methods must respect strike monotonicity, volatility monotonicity, and no-arbitrage bounds.
3. **Delta-hedging validation.** A short call is hedged with the Black-Scholes delta at a finite number of rebalancing dates. The hedging error should be unbiased, shrink like $1/\sqrt M$ in the number of rebalances $M$, and match the Derman-Kamal approximation.
4. **Results and reproducibility.** All tables, figures, and run settings are exported.

The companion notebook, `volatility_validation_submission.ipynb`, studies sensitivity to the volatility input, a two-regime volatility benchmark, and the reliability of simulation-based validation. The two notebooks share the pricing function and model setting but report independent findings. All prices are synthetic and do not establish accuracy in real options markets.

### Model setting

| Quantity | Specification |
|---|---|
| Initial stock price, $S_0$ | 100 dollars |
| Time to expiration, $T$ | 1 year |
| Continuously compounded risk-free rate, $r$ | 4% |
| Baseline volatility, $\sigma$ | 20% annually |
| Strikes | 90, 100, and 110 dollars |
| Simulated terminal prices, $N$ | 100,000 per experiment |
| Hedging experiment | $K = 100$ call; real-world drift $\mu = 8\%$; 4 to 252 rebalances; 20,000 paths |

Option prices and pricing errors are reported in **dollars per share**. Volatilities are annualized and written as decimals in the code.

In [ ]:
from pathlib import Path
import platform
import json

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display

try:
    from bs_model import bs_call
except ModuleNotFoundError as exc:
    if exc.name == "bs_model":
        raise ModuleNotFoundError(
            "bs_model.py is not on the Python import path. "
            "Run this notebook from the project folder that contains it."
        ) from exc
    raise

# ---------------- Financial assumptions ----------------
S0, maturity, rate = 100.0, 1.0, 0.04
sigma_base = 0.20
strikes = [90, 100, 110]
test_volatilities = [0.10, 0.20, 0.30]      # for the volatility-monotonicity test

# Delta-hedging experiment
hedge_strike = 100.0
mu_real = 0.08                               # real-world drift used to simulate hedging paths
rebalance_counts = [4, 12, 52, 252]          # quarterly, monthly, weekly, daily
hedge_paths = 20_000

# ---------------- Simulation settings ----------------
N = 100_000                                  # terminal prices per experiment
SEED_BASELINE = 42                           # Phase 1 and 2 draws
SEED_CONVERGENCE = 2027                      # convergence study
SEED_HEDGE = 2028                            # delta-hedging paths
convergence_budgets = [1_000, 4_000, 16_000, 64_000, 256_000, 1_024_000]
Z_95 = 1.96                                  # two-sided 95% normal quantile
VARIANCE_TOLERANCE = 0.02                    # relative tolerance for the variance check


# ---------------- Output folders ----------------
results_dir = Path("results") / "monte_carlo_verification"
figures_dir = Path("figures") / "monte_carlo_verification"
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
saved_figures = []


def save_figure(fig, name):
    """Export one figure, show it in the notebook, and close it."""
    fig.tight_layout()
    destination = figures_dir / name
    fig.savefig(destination, dpi=200, bbox_inches="tight")
    saved_figures.append(str(destination))
    plt.show()
    plt.close(fig)


versions = {
    "python": platform.python_version(),
    "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
}
print("Environment:", versions)

# Regression check on the shared pricing function (same value as the companion notebook).
assert np.isclose(bs_call(S0, 100, maturity, rate, sigma_base), 9.925053717274437, atol=1e-10, rtol=0)
print("Shared bs_call baseline check passed.")

### Simulation tools

Under the risk-neutral measure the stock follows geometric Brownian motion, $dS_t = rS_t\,dt + \sigma S_t\,dW_t$. Its terminal value has the closed form

$$
S_T = S_0\exp\!\Big(\big(r-\tfrac12\sigma^2\big)T + \sigma\sqrt{T}\,Z\Big), \qquad Z\sim N(0,1),
$$

so a single standard normal draw produces one terminal price and no time stepping is needed. A European call with strike $K$ is priced as the discounted average payoff,

$$
\widehat C = \frac1N\sum_{i=1}^N e^{-rT}\max\big(S_T^{(i)}-K,0\big),
\qquad
SE = \frac{s}{\sqrt N},
\qquad
\widehat C \pm 1.96\,SE,
$$

where $s$ is the sample standard deviation of the discounted payoffs. By the Central Limit Theorem the interval has approximately 95% coverage; it is a normal approximation, not an exact finite-sample guarantee. The functions below implement these formulas once and are reused in every section.

In [ ]:
def simulate_terminal_prices(S0, T, r, sigma, Z):
    """Risk-neutral GBM terminal prices from standard normal draws Z."""
    return S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)


def mc_call(ST, K, T, r):
    """Monte Carlo call price and standard error from terminal prices ST."""
    discounted_payoffs = np.exp(-r * T) * np.maximum(ST - K, 0)
    price = discounted_payoffs.mean()
    se = discounted_payoffs.std(ddof=1) / np.sqrt(len(discounted_payoffs))
    return price, se


def confidence_interval(price, se, z=Z_95):
    """Approximate two-sided normal confidence interval."""
    return price - z * se, price + z * se

## 1. Baseline Black-Scholes verification

Under constant volatility the exact answer is known, so this is where the estimator is tested. The section asks three questions in order: does the simulation produce the right distribution, does it produce the right prices, and does its error shrink at the theoretical rate?

### 1.1 Moments of the terminal price

Using the moment generating function of the normal distribution, $E[e^{aZ}] = e^{a^2/2}$, the lognormal terminal price has

$$
E[S_T] = S_0e^{rT},
\qquad
\operatorname{Var}(S_T) = S_0^2e^{2rT}\big(e^{\sigma^2T}-1\big).
$$

The first identity is the defining property of the risk-neutral measure: the stock grows on average at the risk-free rate. The mean is tested with a $z$-statistic, $z = (\bar S_T - E[S_T])/SE$, and passes when $|z|\le 1.96$. The standard error of a sample variance depends on the fourth moment, so the variance is checked against a simpler relative tolerance of 2%. Checking both matters: an error in the volatility term could leave the mean correct while distorting the spread, and option prices depend directly on the spread.

In [ ]:
rng = np.random.default_rng(SEED_BASELINE)
Z = rng.standard_normal(N)
ST = simulate_terminal_prices(S0, maturity, rate, sigma_base, Z)

theoretical_mean = S0 * np.exp(rate * maturity)
theoretical_var = S0**2 * np.exp(2 * rate * maturity) * (np.exp(sigma_base**2 * maturity) - 1)

simulated_mean = ST.mean()
simulated_var = ST.var(ddof=1)
mean_se = ST.std(ddof=1) / np.sqrt(N)
mean_z = (simulated_mean - theoretical_mean) / mean_se
var_rel_diff = (simulated_var - theoretical_var) / theoretical_var

moments_df = pd.DataFrame([
    {"Moment": "Mean", "Theoretical": theoretical_mean, "Simulated": simulated_mean,
     "Difference": simulated_mean - theoretical_mean,
     "Test statistic": f"z = {mean_z:.3f}", "Pass": abs(mean_z) <= Z_95},
    {"Moment": "Variance", "Theoretical": theoretical_var, "Simulated": simulated_var,
     "Difference": simulated_var - theoretical_var,
     "Test statistic": f"relative diff = {var_rel_diff:.3%}", "Pass": abs(var_rel_diff) < VARIANCE_TOLERANCE},
])
display(moments_df.round(4))

The simulated mean is about 104.010 against a theoretical 104.081, a difference of roughly one standard error ($z\approx -1.07$), which is ordinary sampling variation. The simulated variance is within about 0.8% of its theoretical value of 442.10. The simulated distribution is consistent with the lognormal model on both checks.

### 1.2 Monte Carlo prices against exact Black-Scholes prices

The same terminal prices are used for all three strikes. For each strike the exact Black-Scholes price should fall inside the Monte Carlo 95% confidence interval.

In [ ]:
baseline_rows = []
for K in strikes:
    mc_price, mc_se = mc_call(ST, K, maturity, rate)
    ci_low, ci_high = confidence_interval(mc_price, mc_se)
    bs_price = bs_call(S0, K, maturity, rate, sigma_base)
    baseline_rows.append({
        "Strike": K,
        "BS Price": bs_price,
        "MC Price": mc_price,
        "MC SE": mc_se,
        "CI Lower": ci_low,
        "CI Upper": ci_high,
        "MC minus BS ($)": mc_price - bs_price,
        "z-score": (mc_price - bs_price) / mc_se,
        "BS within CI": ci_low <= bs_price <= ci_high,
    })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(baseline_df["Strike"], baseline_df["BS Price"], "o-", label="Black-Scholes (analytical)")
ax.errorbar(baseline_df["Strike"], baseline_df["MC Price"], yerr=Z_95 * baseline_df["MC SE"],
            fmt="s", capsize=5, label="Monte Carlo (95% CI)")
ax.set(xlabel="Strike ($)", ylabel="Call price ($ per share)",
       title=f"Black-Scholes vs. Monte Carlo, constant volatility {sigma_base:.0%}")
ax.set_xticks(strikes)
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "01_baseline_bs_vs_mc.png")

At all three strikes the exact Black-Scholes price lies inside the individual Monte Carlo 95% interval, with $|z|$ below 0.8 and a largest absolute difference of about 0.04 dollars. The intervals are individual rather than simultaneous, and with three strikes a single miss would not by itself indicate a bug. The error bars in the figure are narrow relative to the prices, which is why the two methods appear to coincide.

### 1.3 Convergence rate

A Monte Carlo estimator should not only be close at one sample size; its error should shrink as $1/\sqrt N$. Each quadrupling of $N$ should therefore halve the standard error, and on log-log axes the standard error should fall on a line of slope $-\tfrac12$. The at-the-money call ($K=100$) is priced at six budgets from 1,000 to about one million draws, each with fresh draws from a separate seed.

In [ ]:
rng_conv = np.random.default_rng(SEED_CONVERGENCE)
bs_atm = bs_call(S0, 100, maturity, rate, sigma_base)

convergence_rows = []
for n in convergence_budgets:
    ST_n = simulate_terminal_prices(S0, maturity, rate, sigma_base, rng_conv.standard_normal(n))
    price, se = mc_call(ST_n, 100, maturity, rate)
    convergence_rows.append({
        "N": n, "MC Price": price, "MC SE": se,
        "Absolute Error ($)": abs(price - bs_atm),
        "Error / SE": abs(price - bs_atm) / se,
    })
convergence_df = pd.DataFrame(convergence_rows)

# Fitted slope of log(SE) against log(N); theory predicts -0.5.
slope = np.polyfit(np.log(convergence_df["N"]), np.log(convergence_df["MC SE"]), 1)[0]
assert -0.55 < slope < -0.45, f"Unexpected convergence slope {slope:.3f}"
display(convergence_df.round(5))
print(f"Fitted log-log slope of SE vs N: {slope:.3f} (theory: -0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(convergence_df["N"], convergence_df["MC SE"], "o-", label="Monte Carlo standard error")
ax.loglog(convergence_df["N"], convergence_df["Absolute Error ($)"], "s", alpha=0.7,
          label="Absolute error vs. Black-Scholes")
reference = convergence_df["MC SE"].iloc[0] * np.sqrt(convergence_df["N"].iloc[0] / convergence_df["N"])
ax.loglog(convergence_df["N"], reference, "--", linewidth=1, label=r"Reference slope $-1/2$")
ax.set(xlabel="Simulated terminal prices, N", ylabel="Dollars per share",
       title="At-the-money call: error shrinks like $1/\\sqrt{N}$")
ax.legend()
ax.grid(alpha=0.25, which="both")
save_figure(fig, "02_convergence.png")

The fitted slope of the standard error is very close to the theoretical $-\tfrac12$. The realized absolute errors scatter around the standard-error line, as expected: the standard error describes the typical size of the error, not its exact value in any single run. The practical consequence is that precision is expensive: one more decimal digit of accuracy requires 100 times as many simulated prices. This is why every price in this notebook is reported together with its standard error.

## 2. Financial consistency tests

Agreement at three strikes is a statistical check. The tests in this section are structural: they verify properties that every valid call price must satisfy, regardless of the model.

| Test | Requirement |
|---|---|
| A. Strike monotonicity | $C(90) > C(100) > C(110)$: a higher strike means a smaller payoff in every state |
| B. Volatility monotonicity | $C(\sigma=10\%) < C(20\%) < C(30\%)$: a call's convex payoff gains value from a wider distribution |
| C. No-arbitrage bounds | $\max\!\big(S_0 - Ke^{-rT},0\big) \le C \le S_0$ |

The tests are applied to both pricing methods because they check two different pieces of code. Applied to the analytical prices, they act as unit tests of `bs_model.py`, catching implementation errors such as a wrong sign in $d_1$. Applied to the Monte Carlo prices, they check the simulation. In expectation Monte Carlo prices satisfy these rules, but each estimate carries random error, so two nearby prices could come out in the wrong order. Using **common random numbers**, the same draws $Z$ for every strike and volatility, makes differences between prices reflect the change in inputs rather than noise. Test B checks only the direction of the price change at three volatilities; how much prices move with the volatility input is studied in the companion notebook.

In [ ]:
def consistency_checks(price_fn, method):
    """Run tests A-C for a pricing function price_fn(K, sigma) and return a summary table."""
    strike_prices = [price_fn(K, sigma_base) for K in strikes]
    vol_prices = [price_fn(100, sigma) for sigma in test_volatilities]
    bounds_ok = all(
        max(S0 - K * np.exp(-rate * maturity), 0) <= price <= S0
        for K, price in zip(strikes, strike_prices)
    )
    return [
        {"Method": method, "Test": "A. Decreasing in strike",
         "Prices": np.round(strike_prices, 4).tolist(), "Pass": bool(np.all(np.diff(strike_prices) < 0))},
        {"Method": method, "Test": "B. Increasing in volatility",
         "Prices": np.round(vol_prices, 4).tolist(), "Pass": bool(np.all(np.diff(vol_prices) > 0))},
        {"Method": method, "Test": "C. No-arbitrage bounds",
         "Prices": np.round(strike_prices, 4).tolist(), "Pass": bounds_ok},
    ]


def analytical_price(K, sigma):
    return bs_call(S0, K, maturity, rate, sigma)


def monte_carlo_price(K, sigma):
    # Common random numbers: the same Z from Section 1 for every strike and volatility.
    return mc_call(simulate_terminal_prices(S0, maturity, rate, sigma, Z), K, maturity, rate)[0]


consistency_df = pd.DataFrame(
    consistency_checks(analytical_price, "Black-Scholes")
    + consistency_checks(monte_carlo_price, "Monte Carlo")
)
assert consistency_df["Pass"].all(), "A financial consistency test failed."
display(consistency_df)

All six checks pass. Prices fall with the strike, rise with volatility, and stay inside the no-arbitrage bounds for both methods. Together with Section 1, this supports the implementation of both pricing methods, so the same engine can be used to test the model's hedge in Section 3. The tests cover the tested inputs only; they are not a proof for all parameter values.

## 3. Delta-hedging validation under discrete rebalancing

Sections 1 and 2 verify the Black-Scholes **price**. A pricing model also makes a stronger claim: the price is the cost of a self-financing strategy that **replicates** the option's payoff. In continuous time, holding $\Delta_t = \partial C/\partial S = N(d_1)$ shares eliminates all risk. In practice a hedge is rebalanced only at discrete dates, so some replication error remains. This section tests whether that error behaves as the theory predicts, which validates the model's hedge ratio rather than its price.

### 3.1 The hedging strategy

A dealer sells one call for the Black-Scholes premium $C_0$ and hedges it over $M$ equally spaced dates $t_k = k\,\Delta t$, with $\Delta t = T/M$:

1. At $t_0$, buy $\Delta_0$ shares and deposit the remainder, $C_0 - \Delta_0S_0$, in a bank account that earns the risk-free rate $r$.
2. At each date $t_k$ before expiry, the bank balance grows by $e^{r\Delta t}$ and the position is rebalanced to the new delta $\Delta_k = N\big(d_1(S_{t_k},T-t_k)\big)$. The cost $(\Delta_k-\Delta_{k-1})S_{t_k}$ is paid from the bank account, so no money is added or withdrawn.
3. At expiry the portfolio is compared with the payoff that must be delivered. The discounted hedging error is

$$
H = e^{-rT}\Big(\Delta_{M-1}S_T + B_T - \max(S_T-K,0)\Big),
$$

where $B_T$ is the final bank balance. A perfect hedge gives $H=0$ on every path.

### 3.2 What the theory predicts

* **Unbiased.** The hedge should cost exactly the premium on average, so $E[H]\approx 0$. The paths are simulated under a **real-world** drift $\mu = 8\%$, not the risk-free rate: the Black-Scholes hedge does not depend on $\mu$, which is why the price does not either.
* **Error shrinks like $1/\sqrt M$.** Each rebalancing interval leaves a small error of order $\Gamma S^2\big((\Delta S/S)^2-\sigma^2\Delta t\big)$. These errors are roughly independent, so their sum has standard deviation proportional to $1/\sqrt M$.
* **Known size.** Derman and Kamal (1999) approximate the standard deviation of the hedging error for an at-the-money option by

$$
\operatorname{sd}(H) \approx \sqrt{\frac{\pi}{4}}\;\frac{\sigma\,\mathcal V}{\sqrt M},
\qquad
\mathcal V = S_0\sqrt T\,\phi(d_1),
$$

where $\mathcal V$ is the Black-Scholes vega. The approximation is asymptotic, so agreement should improve as $M$ grows.

The experiment hedges the $K=100$ call with $M = 4$, 12, 52, and 252 rebalancing dates (quarterly, monthly, weekly, daily) on 20,000 paths each. An unhedged short call is included as a reference.

In [ ]:
def bs_delta(S, K, tau, r, sigma):
    """Black-Scholes call delta N(d1) with time to expiry tau > 0."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    return norm.cdf(d1)


def bs_vega(S, K, T, r, sigma):
    """Black-Scholes vega per unit (decimal) volatility."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)


def delta_hedge_errors(M, n_paths, rng, K=hedge_strike, mu=mu_real, sigma=sigma_base):
    """Discounted terminal hedging error of a short call delta-hedged at M dates."""
    if not isinstance(M, (int, np.integer)) or M < 1:
        raise ValueError("M must be a positive integer.")
    dt = maturity / M
    premium = bs_call(S0, K, maturity, rate, sigma)

    S = np.full(n_paths, S0)
    delta = bs_delta(S, K, maturity, rate, sigma)
    bank = premium - delta * S                       # self-financing: premium funds the hedge

    for k in range(1, M + 1):
        Z_step = rng.standard_normal(n_paths)
        S = S * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z_step)
        bank = bank * np.exp(rate * dt)
        if k < M:                                    # rebalance at every date before expiry
            new_delta = bs_delta(S, K, maturity - k * dt, rate, sigma)
            bank -= (new_delta - delta) * S
            delta = new_delta

    portfolio = delta * S + bank
    return np.exp(-rate * maturity) * (portfolio - np.maximum(S - K, 0))


# Sanity check: with no randomness the bookkeeping must be exact.
# A path that never moves (sigma -> tiny, mu = r) should leave almost no error.
quiet = delta_hedge_errors(52, 10, np.random.default_rng(0), mu=rate, sigma=1e-4)
assert np.allclose(quiet, 0, atol=1e-6), "Self-financing bookkeeping error"
print("Self-financing bookkeeping check passed.")

In [ ]:
rng_hedge = np.random.default_rng(SEED_HEDGE)
vega_atm = bs_vega(S0, hedge_strike, maturity, rate, sigma_base)

hedge_errors = {}
hedge_rows = []
for M in rebalance_counts:
    H = delta_hedge_errors(M, hedge_paths, rng_hedge)
    hedge_errors[M] = H
    sd = H.std(ddof=1)
    mean_se = sd / np.sqrt(hedge_paths)
    dk_sd = np.sqrt(np.pi / 4) * sigma_base * vega_atm / np.sqrt(M)
    hedge_rows.append({
        "Rebalances M": M,
        "Mean error ($)": H.mean(),
        "Mean SE ($)": mean_se,
        "Mean / SE": H.mean() / mean_se,
        "Std of error ($)": sd,
        "Derman-Kamal std ($)": dk_sd,
        "Simulated / formula": sd / dk_sd,
        "5th percentile ($)": np.percentile(H, 5),
        "95th percentile ($)": np.percentile(H, 95),
    })
hedge_df = pd.DataFrame(hedge_rows)

# Reference: the same short call with no hedge at all (premium kept in the bank).
ST_unhedged = simulate_terminal_prices(S0, maturity, mu_real, sigma_base, rng_hedge.standard_normal(hedge_paths))
unhedged = (bs_call(S0, hedge_strike, maturity, rate, sigma_base)
            - np.exp(-rate * maturity) * np.maximum(ST_unhedged - hedge_strike, 0))

hedge_slope = np.polyfit(np.log(hedge_df["Rebalances M"]), np.log(hedge_df["Std of error ($)"]), 1)[0]
assert -0.6 < hedge_slope < -0.4, f"Unexpected hedging convergence slope {hedge_slope:.3f}"
assert (np.abs(hedge_df["Mean error ($)"]) < 0.05 * hedge_df["Std of error ($)"]).all()

display(hedge_df.round(4))
print(f"Unhedged short call: mean {unhedged.mean():.3f}, std {unhedged.std(ddof=1):.3f} dollars")
print(f"Fitted log-log slope of std vs M: {hedge_slope:.3f} (theory: -0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(hedge_df["Rebalances M"], hedge_df["Std of error ($)"], "o-", label="Simulated std of hedging error")
ax.loglog(hedge_df["Rebalances M"], hedge_df["Derman-Kamal std ($)"], "--",
          label=r"Derman-Kamal: $\sqrt{\pi/4}\,\sigma\mathcal{V}/\sqrt{M}$")
ax.set_xticks(rebalance_counts, [f"{m}" for m in rebalance_counts])
ax.set(xlabel="Rebalancing dates per year, M", ylabel="Std of hedging error ($ per share)",
       title="Discrete delta hedging: error shrinks like $1/\\sqrt{M}$")
ax.legend()
ax.grid(alpha=0.25, which="both")
save_figure(fig, "03_hedging_error_convergence.png")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
bins = np.linspace(-4, 4, 121)
for M in [12, 52, 252]:
    ax.hist(hedge_errors[M], bins=bins, histtype="step", linewidth=1.6, density=True,
            label=f"M = {M}  (std {hedge_errors[M].std(ddof=1):.2f})")
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Discounted hedging error ($ per share)", ylabel="Density",
       title="Distribution of hedging error: monthly, weekly, daily rebalancing")
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "04_hedging_error_distribution.png")

### 3.3 Hedging results

The delta hedge behaves as the theory predicts on all three counts.

* **Risk reduction.** An unhedged short call has a hedging error with a standard deviation of about 16.2 dollars per share. Daily delta hedging reduces it to about 0.42 dollars, roughly a 38-fold reduction, using only the Black-Scholes delta.
* **Convergence rate.** The standard deviation falls from about 3.12 dollars with quarterly rebalancing to 0.42 dollars with daily rebalancing. The fitted log-log slope is close to the theoretical $-\tfrac12$: quadrupling the number of rebalancing dates halves the hedging error, the same square-root law as the Monte Carlo error in Section 1.3.
* **Agreement with Derman-Kamal.** The ratio of simulated to predicted standard deviation rises from about 0.92 at $M=4$ to about 0.99 at $M=252$. The formula is an asymptotic approximation, so the closer agreement at frequent rebalancing is expected.

The mean error is close to zero even though the paths use an 8% real-world drift rather than the 4% risk-free rate. At coarse rebalancing there is a small negative mean of about 4 cents, which is statistically detectable with 20,000 paths but below 2.5% of the error's standard deviation; it shrinks to a fraction of a cent at daily rebalancing, consistent with a discretization effect that vanishes as $M\to\infty$. The distributions in the second figure are centered at zero and narrow as $M$ grows, with heavier tails than a normal distribution because errors are largest when the stock ends near the strike, where gamma is highest.

## 4. Discussion and conclusions

The experiments validate two different claims of the Black-Scholes model with the same simulation engine.

The first claim is the price. Under constant volatility, simulated terminal prices reproduce the theoretical lognormal moments, the exact Black-Scholes price lies within the individual Monte Carlo 95% interval at all three strikes, the Monte Carlo standard error decreases at the theoretical $1/\sqrt N$ rate, and both methods pass all financial consistency tests. The evidence supports the implementation of `bs_model.py`.

The second claim is replication. A short call hedged with the Black-Scholes delta and a self-financing bank account recovers its payoff: the mean hedging error is approximately zero under a real-world drift different from the risk-free rate, and its standard deviation falls as $1/\sqrt M$, matching the Derman-Kamal approximation to within about 1% at daily rebalancing. Discrete rebalancing leaves residual risk of about 0.42 dollars per share for daily hedging of an at-the-money one-year call, compared with about 16 dollars unhedged. This residual risk is a property of discrete trading, not an implementation error, and it is present even when every model assumption holds.

### 4.1 Limitations

All prices and paths are synthetic, and the hedging experiment assumes the model is correct: volatility is known and constant, and the stock follows geometric Brownian motion. The experiment ignores transaction costs, bid-ask spreads, and jumps, all of which increase hedging error in practice and would change the optimal rebalancing frequency. Only the at-the-money call is hedged. Confidence intervals are normal approximations and are individual rather than simultaneous. Hedging with a misspecified volatility is a natural extension; it is not studied here because volatility-input sensitivity is the subject of the companion notebook.

## 5. Reproducibility

All financial inputs, seeds, and simulation budgets are set in the first code cell. The pricing function is imported from `bs_model.py` and is not modified. Full-precision tables and the run settings are exported to `results/monte_carlo_verification/`, and figures to `figures/monte_carlo_verification/`. Rounding affects displayed tables only.

In [ ]:
tables_to_save = {
    "moment_checks.csv": moments_df,
    "baseline_results.csv": baseline_df,
    "convergence.csv": convergence_df,
    "consistency_checks.csv": consistency_df,
    "delta_hedging_results.csv": hedge_df,
}
for name, table in tables_to_save.items():
    table.to_csv(results_dir / name, index=False)

run_settings = {
    "S0": S0, "maturity": maturity, "risk_free_rate": rate,
    "baseline_volatility": sigma_base, "strikes": strikes,
    "test_volatilities": test_volatilities,
    "N": N, "convergence_budgets": convergence_budgets,
    "hedging": {
        "strike": hedge_strike, "real_world_drift": mu_real,
        "paths": hedge_paths, "rebalance_counts": rebalance_counts,
        "fitted_slope": float(hedge_slope),
        "unhedged_std": float(unhedged.std(ddof=1)),
    },
    "seeds": {"baseline": SEED_BASELINE, "convergence": SEED_CONVERGENCE, "hedging": SEED_HEDGE},
    "z_95": Z_95, "variance_tolerance": VARIANCE_TOLERANCE,
    "convergence_slope": float(slope),
    "environment": versions,
}
(results_dir / "run_settings.json").write_text(json.dumps(run_settings, indent=2))

print(f"Saved {len(tables_to_save)} tables and run settings to {results_dir}/")
print(f"Saved {len(saved_figures)} figures to {figures_dir}/")

## References

1. Black, F. and Scholes, M. (1973). The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3), 637-654.
2. Derman, E. and Kamal, M. (1999). When you cannot hedge continuously: the corrections to Black-Scholes. *Risk*, 12(1), 82-85.
3. Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering*. Springer. (Chapter 1 for the estimator and its error; Chapter 4 for common random numbers.)
4. Hull, J. C. *Options, Futures, and Other Derivatives*. Pearson. (Black-Scholes formula, delta hedging, and no-arbitrage bounds.)